# 01 - Fill the Database with Fake (Demo) Data
Run cells top to bottom. Each cell creates rows in one part of the database.
Re-running is safe (uses get_or_create).

In [ ]:
import os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'internship_system.settings')
import django
django.setup()

import random, shutil, datetime
from django.conf import settings
from portal.models import *

random.seed(42)   # so the "random" data is the same every run - good for demos

## 1. Academic data (departments, programs, students)

In [ ]:
# Departments
departments = ['Computer Science', 'Software Engineering', 'Artificial Intelligence',
               'Data Science', 'Information Technology', 'Cyber Security']
for name in departments:
    Department.objects.get_or_create(name=name)
print('Departments:', Department.objects.count())

In [ ]:
# Programs (each belongs to a department)
programs = [
    ('BS Computer Science', 'Computer Science', 4),
    ('BS Software Engineering', 'Software Engineering', 4),
    ('BS Artificial Intelligence', 'Artificial Intelligence', 4),
    ('BS Data Science', 'Data Science', 4),
    ('BS Information Technology', 'Information Technology', 4),
]
for prog_name, dept_name, years in programs:
    dept = Department.objects.get(name=dept_name)
    Program.objects.get_or_create(name=prog_name, department=dept, duration_years=years)
print('Programs:', Program.objects.count())

In [ ]:
# Students (our own group members are included, plus more demo students)
students_data = [
    ('Anaya Batool', 'BCSF25M545'),
    ('Muhammad Sarim Talat', 'BCSF25M511'),
    ('Amina Mukhtar', 'BCSF25M547'),
    ('Mahnoor Fatima', 'BCSF25M536'),
    ('Ali Hassan', 'BCSF25M501'),
    ('Fatima Zahra', 'BCSF25M502'),
    ('Usman Khan', 'BCSF25M503'),
    ('Zainab Tariq', 'BCSF25M504'),
    ('Bilal Ahmed', 'BCSF25M505'),
    ('Hira Shahid', 'BCSF25M506'),
]
program_list = list(Program.objects.all())
for name, roll in students_data:
    email = roll.lower() + '@student.edu.pk'
    cgpa = round(random.uniform(2.60, 3.95), 2)
    program = random.choice(program_list)
    Student.objects.get_or_create(roll_no=roll, defaults={
        'name': name, 'email': email, 'cgpa': cgpa, 'program': program})
print('Students:', Student.objects.count())

## 2. Skills

In [ ]:
skills = ['Python', 'Java', 'SQL', 'HTML/CSS', 'JavaScript', 'Django',
          'Oracle Database', 'Communication', 'Teamwork', 'Git']
for name in skills:
    Skill.objects.get_or_create(name=name)
print('Skills:', Skill.objects.count())

In [ ]:
# Give every student 3-5 random skills
all_students = list(Student.objects.all())
all_skills = list(Skill.objects.all())
levels = ['Beginner', 'Intermediate', 'Advanced']
for st in all_students:
    chosen = random.sample(all_skills, k=random.randint(3, 5))
    for sk in chosen:
        StudentSkill.objects.get_or_create(
            student=st, skill=sk,
            defaults={'proficiency': random.choice(levels)})
print('Student-skill rows:', StudentSkill.objects.count())

## 3. Companies, contacts and internships

In [ ]:
companies_data = [
    ('TechNest Solutions', 'Software House'),
    ('DataBridge Systems', 'Data Analytics'),
    ('CloudSphere Inc.', 'Cloud Computing'),
    ('CodeCrafters', 'Software House'),
    ('FinEdge Technologies', 'Fintech'),
    ('HealthSync', 'Healthcare IT'),
    ('EduSoft', 'EdTech'),
    ('GreenGrid Energy', 'Energy IT'),
]
for name, industry in companies_data:
    Company.objects.get_or_create(name=name, defaults={
        'industry': industry,
        'website': 'https://' + name.lower().replace(' ', '').replace('.', '') + '.com',
        'address': 'Lahore, Pakistan'})
print('Companies:', Company.objects.count())

In [ ]:
# One or two recruiter contacts per company
first_names = ['Sara', 'Omar', 'Kiran', 'Danish', 'Nadia', 'Hamza']
last_names = ['Malik', 'Raza', 'Sheikh', 'Butt', 'Ali', 'Farooq']
for comp in Company.objects.all():
    n = random.randint(1, 2)
    for i in range(n):
        fn, ln = random.choice(first_names), random.choice(last_names)
        CompanyContact.objects.get_or_create(
            company=comp, email=(fn + ln).lower() + '@' + comp.name.lower().replace(' ', '') + '.com',
            defaults={'person_name': fn + ' ' + ln, 'phone': '03' + str(random.randint(100000000, 499999999))})
print('Contacts:', CompanyContact.objects.count())

In [ ]:
# Internships
today = datetime.date.today()
internships_data = [
    ('Software Development Intern', 'TechNest Solutions', 30000, 60),
    ('Database Intern', 'DataBridge Systems', 25000, 45),
    ('Frontend Intern', 'CodeCrafters', 20000, 30),
    ('QA / Testing Intern', 'TechNest Solutions', 15000, 40),
    ('Data Analyst Intern', 'DataBridge Systems', 35000, 55),
    ('Cloud Intern', 'CloudSphere Inc.', 40000, 70),
    ('Web Developer Intern', 'EduSoft', 18000, 35),
    ('Python Backend Intern', 'FinEdge Technologies', 32000, 50),
    ('IT Support Intern', 'GreenGrid Energy', 12000, 25),
    ('ML Intern', 'HealthSync', 45000, 65),
]
for title, comp_name, stipend, days in internships_data:
    comp = Company.objects.get(name=comp_name)
    Internship.objects.get_or_create(title=title, company=comp, defaults={
        'description': title + ': hands-on work with mentors, real projects, weekly reviews.',
        'location': random.choice(['Lahore', 'Karachi', 'Islamabad', 'Remote']),
        'stipend': stipend,
        'deadline': today + datetime.timedelta(days=days),
        'status': 'Open'})
print('Internships:', Internship.objects.count())

In [ ]:
# Required skills per internship (2-4 random skills each)
for ip in Internship.objects.all():
    for sk in random.sample(all_skills, k=random.randint(2, 4)):
        InternshipSkill.objects.get_or_create(internship=ip, skill=sk)
print('Internship-skill rows:', InternshipSkill.objects.count())

## 4. Resumes (the fake PDF files included in the project)

In [ ]:
# Copy the fake PDF resumes into media/resumes and link them to students
fake_dir = os.path.join(settings.BASE_DIR, 'fake_resumes')
media_resumes = os.path.join(settings.MEDIA_ROOT, 'resumes')
os.makedirs(media_resumes, exist_ok=True)

pdf_files = sorted(os.listdir(fake_dir))
for i, filename in enumerate(pdf_files):
    src = os.path.join(fake_dir, filename)
    with open(src, 'rb') as f:
        student = all_students[i % len(all_students)]
        Resume.objects.get_or_create(student=student, title=filename,
            defaults={'file': 'resumes/' + filename})
        # physically copy the file to media/resumes/
        shutil.copy(src, os.path.join(media_resumes, filename))
print('Resumes:', Resume.objects.count())

## 5. Applications + their status history

In [ ]:
# The possible application stages (our lookup table)
for s in ['Submitted', 'Shortlisted', 'Interview', 'Offered', 'Rejected']:
    ApplicationStatus.objects.get_or_create(name=s)

# A realistic "progress chain" per application
chains = [
    ['Submitted'],
    ['Submitted', 'Shortlisted'],
    ['Submitted', 'Shortlisted', 'Interview'],
    ['Submitted', 'Shortlisted', 'Interview', 'Offered'],
    ['Submitted', 'Shortlisted', 'Interview', 'Rejected'],
]
open_ips = list(Internship.objects.filter(status='Open'))
resumes = {r.student_id: r for r in Resume.objects.all()}

created = 0
for st in all_students:
    for ip in random.sample(open_ips, 3):   # each student applies to 3 internships
        if Application.objects.filter(student=st, internship=ip).exists():
            continue
        app = Application.objects.create(
            student=st, internship=ip,
            resume=resumes.get(st.id),
            cover_letter='I am interested in this internship and eager to learn.')
        for stage in random.choice(chains):
            ApplicationStatusHistory.objects.create(
                application=app,
                status=ApplicationStatus.objects.get(name=stage),
                note='seeded demo data')
        created += 1
print('New applications:', created, '| Total:', Application.objects.count())

## 6. Interviews, Offers and Placements

In [ ]:
# Interview for every application that reached the 'Interview' stage
reached_interview = ApplicationStatusHistory.objects.filter(
    status__name='Interview').values_list('application_id', flat=True)

for app_id in reached_interview:
    app = Application.objects.get(id=app_id)
    Interview.objects.get_or_create(application=app, defaults={
        'interview_date': datetime.datetime.combine(
            today + datetime.timedelta(days=10), datetime.time(10, 0)),
        'mode': random.choice(['Online', 'Onsite']),
        'result': random.choice(['Pending', 'Passed', 'Failed'])})
print('Interviews:', Interview.objects.count())

In [ ]:
# Offer + Placement for every application that reached 'Offered'
reached_offer = ApplicationStatusHistory.objects.filter(
    status__name='Offered').values_list('application_id', flat=True)

for app_id in reached_offer:
    app = Application.objects.get(id=app_id)
    offer, _ = Offer.objects.get_or_create(application=app, defaults={
        'stipend': app.internship.stipend or 20000, 'status': 'Accepted'})
    Placement.objects.get_or_create(offer=offer, defaults={
        'start_date': today + datetime.timedelta(days=30),
        'end_date': today + datetime.timedelta(days=120),
        'status': 'Ongoing'})
print('Offers:', Offer.objects.count(), '| Placements:', Placement.objects.count())

## Done!
Now open **http://127.0.0.1:8000** to see the website, or **/admin** to browse/edit
every table (login: admin / admin123).